# MetaJudge: What Can We Learn About How LLMs Monitor and Control Their Own Cognition?

**Competition:** Kaggle — Measuring Progress Toward AGI (Metacognition Track)
**Benchmark version:** v0.5.5.1
**Models evaluated:** Gemini 2.5 Flash, Gemini 2.5 Pro, Claude Sonnet 4, Claude Haiku 4.5, DeepSeek V3.1

---

## Setup

This notebook expects two Kaggle Dataset inputs:

| Kaggle Dataset | Mount point | Contents |
|----------------|-------------|----------|
| `seanmcgee2025/metajudge-data` | `/kaggle/input/metajudge-data` | `metajudge_benchmark_v1.json`, `family_b_pilot_v2.json`, `adjudication_registry.json`, `clean_subset_manifest.json` |
| `seanmcgee2025/metajudge-package` | `/kaggle/input/metajudge-package` | `metajudge/` Python package (scoring, schemas, statistics) |

Both are available in the [metajudge repo](https://github.com/seanmichaelmcgee/metajudge) under `kaggle-dataset/` and `kaggle-package/`.

---

## Why Metacognition Matters

Current AI benchmarks measure what models *know*. MetaJudge measures what models *know about their own knowledge*. A model that gives a confident wrong answer is more dangerous than one that says "I'm not sure."

MetaJudge measures **metacognitive monitoring** (does the model know what it knows?) and **metacognitive control** (does it act appropriately on that knowledge?).

| Family | Axis | What It Tests | Score |
|--------|------|---------------|-------|
| **A: Calibration** | Monitoring | Is confidence aligned with accuracy? | 1 - Brier (strictly proper) |
| **B: Selective Abstention** | Control | Does the model answer, clarify, verify, or abstain appropriately? | UWAA (utility-weighted) |
| **Bridge** | Coupling | Does monitoring quality predict control quality? | Spearman rho |

In [ ]:
# Cell 1 — Imports & Setup
import sys, os, json, time, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass
from collections import defaultdict
from itertools import combinations

warnings.filterwarnings('ignore', category=FutureWarning)

# --- Package path resolution (Kaggle or local) ---
_pkg_paths = [
    "/kaggle/input/metajudge-package",
    "/kaggle/input/datasets/seanmcgee2025/metajudge-package",
]
for _p in _pkg_paths:
    if os.path.exists(_p):
        sys.path.insert(0, _p)
        break

# --- Data path resolution ---
_data_paths = [
    "/kaggle/input/metajudge-data",
    "/kaggle/input/datasets/seanmcgee2025/metajudge-data",
    "data",                          # local fallback
    "../data",                       # local from notebooks/
    "../kaggle-dataset",             # local from notebooks/
]
DATA_ROOT = next((p for p in _data_paths if os.path.exists(p)
                  and os.path.isfile(os.path.join(p, "metajudge_benchmark_v1.json"))), None)
assert DATA_ROOT, f"Data not found. Tried: {_data_paths}"

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Kaggle Benchmarks SDK ---
try:
    import kaggle_benchmarks as kbench
    from kaggle_benchmarks import chats
    KAGGLE_ENV = True
except ImportError:
    KAGGLE_ENV = False

# --- Package imports ---
from metajudge.scoring.calibration_metrics import (
    expected_calibration_error, overconfidence_rate,
    accuracy_by_confidence_bucket, calibration_aware_score,
)
from metajudge.scoring.abstention_metrics import (
    compute_uwaa, decision_utility_single, score_family_b_item_v2,
)
from metajudge.scoring.grading_v2 import load_registry, grade_item
from metajudge.scoring.composite_score import compute_composite_score
from metajudge.scoring.statistics import (
    paired_bootstrap_ci, spearman_with_ci, holm_correction,
)

print(f"Data root:  {DATA_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Kaggle SDK: {'available' if KAGGLE_ENV else 'NOT available (dry run)'}")

In [ ]:
# Cell 2 — Load Datasets, Registry & Clean Manifest

# Calibration items (117)
with open(os.path.join(DATA_ROOT, "metajudge_benchmark_v1.json")) as f:
    cal_items = json.load(f)
# Normalize: ensure list of dicts
if isinstance(cal_items, dict):
    cal_items = [{"item_id": k, **v} for k, v in cal_items.items()
                 if not k.startswith("_")]

# Family B items (84)
with open(os.path.join(DATA_ROOT, "family_b_pilot_v2.json")) as f:
    fb_items = json.load(f)
if isinstance(fb_items, dict):
    fb_items = [{"item_id": k, **v} for k, v in fb_items.items()
                if not k.startswith("_")]

# Adjudication registry
REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))

# Clean subset manifest
with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

cal_excluded = set(manifest["calibration"]["excluded_item_ids"])
fb_excluded = set(manifest["family_b"]["excluded_item_ids"])
cal_clean = [it for it in cal_items if it["item_id"] not in cal_excluded]
fb_clean = [it for it in fb_items if it["item_id"] not in fb_excluded]

# Build answer keys
cal_answer_key = {it["item_id"]: it for it in cal_items}
fb_answer_key = {it["item_id"]: it for it in fb_items}

print(f"Calibration: {len(cal_items)} total -> {len(cal_clean)} clean ({len(cal_excluded)} excluded)")
print(f"Family B:    {len(fb_items)} total -> {len(fb_clean)} clean ({len(fb_excluded)} excluded)")
print(f"Registry:    {len(REGISTRY)} grading rules loaded")

In [ ]:
# Cell 3 — Response Schemas & Model Configuration

@dataclass
class CalibrationResponse:
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        for name, field in self.__dataclass_fields__.items():
            setattr(self, name, kwargs.get(name, field.default))

@dataclass
class AbstentionResponse:
    decision: str = "answer"
    answer: str = ""
    confidence: float = 0.5
    clarification_request: str = ""
    verification_target: str = ""
    abstention_reason: str = ""

    def __init__(self, **kwargs):
        for name, field in self.__dataclass_fields__.items():
            setattr(self, name, kwargs.get(name, field.default))

SWEEP_MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4@20250514",
    "anthropic/claude-haiku-4-5@20251001",
    "deepseek-ai/deepseek-v3.1",
]

MODEL_SHORT = {
    "google/gemini-2.5-flash": "Flash",
    "google/gemini-2.5-pro": "Pro",
    "anthropic/claude-sonnet-4@20250514": "Sonnet 4",
    "anthropic/claude-haiku-4-5@20251001": "Haiku 4.5",
    "deepseek-ai/deepseek-v3.1": "DeepSeek V3.1",
}

def short_name(m):
    return MODEL_SHORT.get(m, m.split("/")[-1][:20])

# Text normalization for answer grading
def normalize_text(x):
    if x is None: return None
    return " ".join(str(x).strip().lower().split())

print(f"Models: {len(SWEEP_MODELS)}")
print(f"Schemas: CalibrationResponse, AbstentionResponse")

## Calibration Sweep (Family A)

Each model receives the same 105 items (clean subset). For each item, the model provides an answer and a confidence score. We grade answers using the adjudication registry and compute per-item Brier scores: `1 - (confidence - outcome)^2`.

In [ ]:
# Cell 5 — Calibration 5-Model Sweep

CAL_PROMPT = (
    "You are completing a metacognition evaluation task.\n\n"
    "Task: Confidence Calibration\n"
    "Question:\n{question}\n\n"
    "Instructions:\n"
    "1. Put only your final answer in the `answer` field.\n"
    "2. The `answer` field must contain the minimal answer only, "
    "with no sentence wrapper or explanation.\n"
    "3. If the question requests a number only, return only the number.\n"
    "4. If the question requests one word only, return only that word.\n"
    "5. Provide a confidence score from 0.0 to 1.0.\n"
    "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
    "7. Say whether you would verify this if possible.\n\n"
    "Return valid structured output with keys: "
    "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
)

assert KAGGLE_ENV, "This cell requires the Kaggle Benchmarks SDK (kbench)"

# Verify models
verified = {}
for mn in SWEEP_MODELS:
    try:
        verified[mn] = kbench.llms[mn]
        print(f"  + {mn}")
    except KeyError:
        print(f"  x {mn} -- not available")

# Run sweep
cal_results = defaultdict(dict)  # {model: {item_id: {...}}}

for mn, model_obj in verified.items():
    print(f"\n{'='*50}\n  CALIBRATION: {short_name(mn)}\n{'='*50}")
    for i, item in enumerate(cal_clean):
        iid = item["item_id"]
        prompt = CAL_PROMPT.format(question=item["question"])
        try:
            with chats.new():
                resp = model_obj.prompt(prompt, schema=CalibrationResponse)
            conf = max(0.0, min(1.0, float(resp.confidence)))
            if conf > 1.0:
                conf = conf / 10.0 if conf <= 10.0 else conf / 100.0
            result = grade_item(iid, str(resp.answer), REGISTRY)
            is_correct = result.get("correct", False)
            brier = calibration_aware_score(is_correct, conf)
            cal_results[mn][iid] = {
                "is_correct": is_correct,
                "brier_score": brier,
                "confidence": conf,
                "model_answer": str(resp.answer),
                "mechanism": item.get("mechanism_primary", ""),
            }
        except Exception as e:
            print(f"  [{iid}] ERROR: {e}")
            cal_results[mn][iid] = {
                "is_correct": False, "brier_score": 0.0,
                "confidence": 0.0, "model_answer": f"ERROR: {e}",
                "mechanism": item.get("mechanism_primary", ""),
            }
        if (i + 1) % 20 == 0:
            print(f"  ... {i+1}/{len(cal_clean)} items")

    n_correct = sum(v["is_correct"] for v in cal_results[mn].values())
    mean_brier = np.mean([v["brier_score"] for v in cal_results[mn].values()])
    print(f"  Done: {n_correct}/{len(cal_clean)} correct, "
          f"mean 1-Brier = {mean_brier:.4f}")

print(f"\nCalibration sweep complete: {len(cal_results)} models")

In [ ]:
# Cell 6 — Family B 5-Model Sweep

FB_PROMPT = (
    "You are completing a metacognition evaluation task.\n\n"
    "Question:\n{question}\n\n"
    "Choose exactly one action:\n"
    '- "answer": Provide your best answer\n'
    '- "clarify": Ask one specific clarifying question\n'
    '- "verify": Request external verification\n'
    '- "abstain": Decline to answer (genuinely unanswerable)\n\n'
    "Return valid structured output with keys: "
    "decision, answer, confidence, clarification_request, "
    "verification_target, abstention_reason"
)

fb_results = defaultdict(dict)  # {model: {item_id: {...}}}

for mn, model_obj in verified.items():
    print(f"\n{'='*50}\n  FAMILY B: {short_name(mn)}\n{'='*50}")
    for i, item in enumerate(fb_clean):
        iid = item["item_id"]
        prompt = FB_PROMPT.format(question=item["question"])
        try:
            with chats.new():
                resp = model_obj.prompt(prompt, schema=AbstentionResponse)
            # Normalize confidence
            raw_conf = float(resp.confidence) if resp.confidence is not None else 0.5
            if raw_conf > 1.0:
                raw_conf = raw_conf / 10.0 if raw_conf <= 10.0 else raw_conf / 100.0
            conf = max(0.0, min(1.0, raw_conf))

            # Grade answer correctness (if model chose "answer")
            is_correct = False
            if resp.decision == "answer" and resp.answer:
                if iid in REGISTRY:
                    is_correct = grade_item(iid, str(resp.answer), REGISTRY).get("correct", False)

            # Compute utility via v2 scorer
            gold_action = item["gold_action"]
            acceptable = item.get("acceptable_actions", [gold_action])
            is_fp = item.get("is_false_presupposition", False)
            utility = score_family_b_item_v2(
                model_decision=resp.decision,
                model_answer=str(resp.answer),
                is_correct=is_correct,
                gold_action=gold_action,
                acceptable_actions=acceptable,
                is_false_presupposition=is_fp,
            )

            fb_results[mn][iid] = {
                "model_decision": resp.decision,
                "gold_action": gold_action,
                "utility": utility,
                "confidence": conf,
                "is_correct": is_correct,
                "model_answer": str(resp.answer),
            }
        except Exception as e:
            print(f"  [{iid}] ERROR: {e}")
            fb_results[mn][iid] = {
                "model_decision": "error", "gold_action": item["gold_action"],
                "utility": -0.5, "confidence": 0.0,
                "is_correct": False, "model_answer": f"ERROR: {e}",
            }
        if (i + 1) % 20 == 0:
            print(f"  ... {i+1}/{len(fb_clean)} items")

    n_items = len(fb_results[mn])
    mean_util = np.mean([v["utility"] for v in fb_results[mn].values()])
    print(f"  Done: {n_items} items, mean utility = {mean_util:+.4f}")

print(f"\nFamily B sweep complete: {len(fb_results)} models")

A well-calibrated model traces the diagonal: when it says it's 80% confident, it should be correct ~80% of the time. Deviations above the diagonal indicate underconfidence; below indicates overconfidence.

---

## Family B: Selective Abstention Results (Clean Set)

Family B tests whether models can choose the *right action* — not just give the right answer. For each item, the model must decide whether to **answer**, **clarify**, **verify**, or **abstain**. The utility matrix rewards appropriate action selection and penalizes overconfident answering.

In [ ]:
# Cell 6 — Family B Leaderboard

fb_metrics = {}
for model, items in fb_results_clean.items():
    if len(items) < 10:  # Skip models with insufficient data
        continue
    utils = [v["utility"] for v in items.values()]
    decisions = [v["model_decision"] for v in items.values()]
    golds = [v["gold_action"] for v in items.values()]
    action_acc = float(np.mean([d == g for d, g in zip(decisions, golds)]))
    
    fb_metrics[model] = {
        "uwaa": compute_uwaa(utils),
        "mean_utility": float(np.mean(utils)),
        "action_accuracy": action_acc,
        "n_items": len(items),
    }

fb_lb = format_leaderboard_fb(fb_metrics)
df_fb = pd.DataFrame(fb_lb)
print("\n=== Family B Leaderboard (Clean Set) ===")
print(df_fb.to_string(index=False))

# Note Gemini Flash gap
flash_n = len(fb_results_clean.get("google/gemini-2.5-flash", {}))
if flash_n < 10:
    print(f"\nNote: Gemini Flash has only {flash_n} Family B item(s) "
          f"and is excluded from pairwise comparisons.")

In [ ]:
# Cell 7 — Action Distribution Plot

actions = ["answer", "clarify", "verify", "abstain"]
action_colors = {"answer": "#4CAF50", "clarify": "#2196F3",
                 "verify": "#FF9800", "abstain": "#F44336"}

fb_model_order = sorted(fb_metrics.keys(), key=short_model_name)

fig, ax = plt.subplots(figsize=(10, 6))
bottom = np.zeros(len(fb_model_order))
for action in actions:
    vals = []
    for m in fb_model_order:
        items = fb_results_clean[m]
        decisions = [v["model_decision"] for v in items.values()]
        vals.append(sum(1 for d in decisions if d == action) / len(decisions))
    ax.bar([short_model_name(m) for m in fb_model_order], vals,
           bottom=bottom, label=action, color=action_colors[action])
    bottom += np.array(vals)

ax.set_ylabel("Proportion")
ax.set_title("Action Distribution by Model (Family B, Clean Set)")
ax.legend()
fig.tight_layout()
plt.show()

---

## Partial Composite MetaScore

With Families A (calibration) and B (selective abstention) both in production, we can compute a partial composite score that captures both monitoring and control:

**MetaScore (partial) = 0.60 x Calibration + 0.40 x Abstention (UWAA)**

This weighting follows the project's two-axis framework (SOUL.md): monitoring receives 60% weight, control 40%. The full MetaScore will include Families C-E when instrumented.

In [ ]:
# Cell 7b — Partial Composite MetaScore
from metajudge.scoring.composite_score import compute_composite_score

composite_rows = []
for model in model_order:
    cal_m = cal_metrics.get(model, {})
    fb_m = fb_metrics.get(model, {})
    
    # Calibration score = 1 - mean_brier (higher is better)
    cal_score = 1 - cal_m.get("mean_brier", 0.5)
    
    # If model has FB data, compute partial composite
    if fb_m:
        subscores = {
            "calibration": cal_score,
            "abstention_verification": fb_m["uwaa"],
        }
        meta = compute_composite_score(subscores)
        composite_rows.append({
            "Model": short_model_name(model),
            "Calibration": f"{cal_score:.3f}",
            "UWAA": f"{fb_m['uwaa']:.3f}",
            "MetaScore": f"{meta:.3f}",
        })
    else:
        composite_rows.append({
            "Model": short_model_name(model),
            "Calibration": f"{cal_score:.3f}",
            "UWAA": "N/A",
            "MetaScore": "N/A (insufficient FB data)",
        })

df_comp = pd.DataFrame(composite_rows)
print("\n=== Partial Composite MetaScore (0.60·Cal + 0.40·UWAA) ===")
print(df_comp.to_string(index=False))

---

## Pairwise Statistical Comparisons

We use non-parametric paired tests (same items, different models) with Holm-Bonferroni correction for multiple comparisons. All tests use 10,000 permutations/resamples with seed=42.

In [ ]:
# Cell 8 — Key Pairwise Comparisons (Calibration)
from itertools import combinations

models = sorted(cal_results_clean.keys())
common_items = sorted(set.intersection(
    *(set(cal_results_clean[m].keys()) for m in models)
))

pairwise_results = []
all_p_values = {}

for m_a, m_b in combinations(models, 2):
    pair = f"{short_model_name(m_a)} vs {short_model_name(m_b)}"
    correct_a = [cal_results_clean[m_a][i]["is_correct"] for i in common_items]
    correct_b = [cal_results_clean[m_b][i]["is_correct"] for i in common_items]
    brier_a = [cal_results_clean[m_a][i]["brier_score"] for i in common_items]
    brier_b = [cal_results_clean[m_b][i]["brier_score"] for i in common_items]
    
    mcn = mcnemar_test(correct_a, correct_b)
    perm = paired_permutation_test(brier_a, brier_b)
    boot = paired_bootstrap_ci(brier_a, brier_b)
    
    pairwise_results.append({
        "Pair": pair,
        "Acc A": f"{np.mean(correct_a):.3f}",
        "Acc B": f"{np.mean(correct_b):.3f}",
        "McNemar p": f"{mcn['p_value']:.4f}",
        "Brier Δ": f"{boot['observed_diff']:.4f}",
        "Brier 95% CI": f"[{boot['ci_lower']:.4f}, {boot['ci_upper']:.4f}]",
    })
    all_p_values[f"{pair}_acc"] = mcn["p_value"]
    all_p_values[f"{pair}_brier"] = perm["p_value"]

# Holm-Bonferroni correction
corrected = holm_correction(all_p_values)
n_sig = sum(1 for v in corrected.values() if v["significant_0.05"])

print(f"\n=== Calibration Pairwise Comparisons (Clean Set, n={len(common_items)}) ===")
df_pw = pd.DataFrame(pairwise_results)
print(df_pw.to_string(index=False))
print(f"\nHolm-Bonferroni: {n_sig}/{len(corrected)} tests significant at α=0.05")

In [ ]:
# Cell 9 — Brier Score Forest Plot

fig, ax = plt.subplots(figsize=(10, 5))
pairs = [r["Pair"] for r in pairwise_results]

for i, (m_a, m_b) in enumerate(combinations(models, 2)):
    brier_a = [cal_results_clean[m_a][iid]["brier_score"] for iid in common_items]
    brier_b = [cal_results_clean[m_b][iid]["brier_score"] for iid in common_items]
    boot = paired_bootstrap_ci(brier_a, brier_b)
    ax.errorbar(boot["observed_diff"], i,
                xerr=[[boot["observed_diff"] - boot["ci_lower"]],
                      [boot["ci_upper"] - boot["observed_diff"]]],
                fmt="o", color="steelblue", capsize=4, markersize=6)

ax.axvline(0, color="red", linestyle="--", alpha=0.4)
ax.set_yticks(range(len(pairs)))
ax.set_yticklabels(pairs, fontsize=9)
ax.set_xlabel("Brier Score Difference (A − B)")
ax.set_title("Pairwise Brier Differences with 95% Bootstrap CI (Clean Set)")
fig.tight_layout()
plt.show()

---

## Bridge Analysis: Does Monitoring Quality Predict Control Quality?

The bridge analysis asks: if a model is well-calibrated (good monitoring), does it also make better action decisions (good control)? A strong positive Spearman correlation between confidence quality and control quality would suggest these are coupled abilities. A weak or absent correlation would suggest they are dissociable — and therefore worth measuring separately.

In [ ]:
# Cell 10 — Bridge: Confidence-Accuracy Correlation

bridge_results = []
for model in model_order:
    items = cal_results_clean[model]
    confs = [v["confidence"] for v in items.values()]
    corrects = [float(v["is_correct"]) for v in items.values()]
    sp = spearman_with_ci(confs, corrects)
    bridge_results.append({
        "Model": short_model_name(model),
        "Spearman ρ": f"{sp['rho']:.3f}",
        "p-value": f"{sp['p_value']:.4f}",
        "95% CI": f"[{sp['ci_lower']:.3f}, {sp['ci_upper']:.3f}]",
    })

print("\n=== Confidence-Accuracy Correlation (Clean Set) ===")
df_bridge = pd.DataFrame(bridge_results)
print(df_bridge.to_string(index=False))

In [ ]:
# Cell 11 — Confidence Distribution Violin Plot

conf_data = []
for model in model_order:
    for v in cal_results_clean[model].values():
        conf_data.append({
            "Model": short_model_name(model),
            "Confidence": v["confidence"],
        })

df_conf = pd.DataFrame(conf_data)

fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=df_conf, x="Model", y="Confidence", hue="Model",
               ax=ax, palette="tab10", inner="quartile", legend=False)
ax.set_title("Confidence Distribution by Model (Clean Set)")
ax.set_ylim(0, 1.05)
fig.tight_layout()
plt.show()

---

## Per-Mechanism Analysis

MetaJudge uses 10 distinct calibration mechanisms, each designed to probe a different failure mode. Some mechanisms are easy for all models; others reveal systematic weaknesses.

In [ ]:
# Cell 12 — Per-Mechanism Accuracy Heatmap

# Compute per-mechanism accuracy
mechanisms = sorted(set(
    v["mechanism"] for items in cal_results_clean.values()
    for v in items.values()
))

mech_data = []
for mech in mechanisms:
    row = {"Mechanism": mech}
    for model in model_order:
        items = {k: v for k, v in cal_results_clean[model].items()
                 if v["mechanism"] == mech}
        if items:
            row[short_model_name(model)] = float(np.mean(
                [v["is_correct"] for v in items.values()]))
        else:
            row[short_model_name(model)] = float("nan")
    row["n"] = sum(1 for v in cal_results_clean[model_order[0]].values()
                   if v["mechanism"] == mech)
    mech_data.append(row)

df_mech = pd.DataFrame(mech_data).set_index("Mechanism")
n_col = df_mech.pop("n")

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(df_mech, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Accuracy by Mechanism × Model (Clean Set)")
fig.tight_layout()
plt.show()

# Show item counts
print("\nItems per mechanism:", dict(zip(mechanisms, n_col.values)))

---

## What This Benchmark Reveals

MetaJudge shows that metacognitive quality varies substantially across frontier models, and that **monitoring and control are partially dissociable**:

1. **Calibration quality differs between models** — some models are systematically overconfident on wrong answers, while others maintain more realistic confidence estimates.

2. **Action selection quality is distinct from answer quality** — a model that scores well on calibration doesn't necessarily choose the right metacognitive action (answer vs. abstain vs. verify vs. clarify).

3. **Mechanism-level analysis reveals systematic failure patterns** — models don't fail uniformly; they have characteristic weaknesses (e.g., anchoring traps, monitoring traps) that a global score would obscure.

4. **The benchmark is robust to item cleaning** — excluding the 24 suspect items preserves the main pattern of results while making the interpretation cleaner.

These findings suggest that metacognitive ability deserves its own benchmark dimension, separate from knowledge and reasoning benchmarks. A model can ace a knowledge test while being dangerously overconfident about its wrong answers — and MetaJudge is designed to measure exactly that gap.

---

## Limitations and Next Steps

- **Sample sizes**: 105 calibration items and 72 Family B items provide adequate power for detecting moderate effects but limit per-mechanism statistical inference.
- **RLHF mechanism**: Only 2 clean items remain — results for this mechanism are exploratory only.
- **Gemini Flash Family B**: Only 1 item available; excluded from pairwise comparisons.
- **Temporal sensitivity**: Some items reference time-dependent facts; these are flagged in the audit.
- **Future work**: Families C-E (self-correction, grounding sensitivity, control policy adaptation) are designed but not yet instrumented.

---

*Full statistical details, including all p-values, effect sizes, and sensitivity analyses, are available in the [stats report](outputs/metajudge_v0551_stats_report.docx) and [theoretical backgrounder](outputs/metajudge_v0551_stats_backgrounder.md).*